In [4]:
import csv
import numpy as np
from dataclasses import dataclass
from typing import List, Dict
from gensim.models import KeyedVectors

@dataclass
class Category:
    label: str
    words: List[str]
    difficulty: int

@dataclass
class ConnectionsPuzzle:
    puzzle_id: int
    categories: List[Category]

    @property
    def raw_board(self) -> List[str]:
        """Returns the flattened list of 16 words as the solver sees them."""
        return [word.lower() for cat in self.categories for word in cat.words]

    def get_ground_truth_matrix(self) -> np.ndarray:
        """
        Creates a 16x16 binary matrix representing the correct solution.
        1 if words i and j belong to the same category, 0 otherwise.
        """
        matrix = np.zeros((16, 16), dtype=int)
        for i in range(4):
            start_idx = i * 4
            matrix[start_idx:start_idx+4, start_idx:start_idx+4] = 1
        return matrix

In [5]:
def load_puzzles_from_csv(filepath: str) -> List[ConnectionsPuzzle]:
    difficulty_map = {
        "easy": 1,
        "medium": 2,
        "hard": 3,
        "very hard": 4
    }
    
    puzzles = []
    
    with open(filepath, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader, None) # Skip header
        
        all_rows = list(reader)
        puzzle_counter = 1
        
        # Group every 4 rows into a single puzzle
        for i in range(0, len(all_rows), 4):
            game_rows = all_rows[i:i+4]
            
            if len(game_rows) != 4:
                print(f"Warning: Incomplete puzzle found at row {i}. Skipping.")
                break
                
            categories = []
            for row in game_rows:
                # Extract your columns based on your layout
                words = [row[0].strip(), row[1].strip(), row[2].strip(), row[3].strip()]
                label = row[4].strip()
                difficulty_str = row[5].strip().lower()
                diff_int = difficulty_map.get(difficulty_str, 0)
                
                categories.append(Category(label=label, words=words, difficulty=diff_int))
            
            puzzle = ConnectionsPuzzle(puzzle_id=puzzle_counter, categories=categories)
            puzzles.append(puzzle)
            puzzle_counter += 1
            
    return puzzles

In [8]:
from gensim.scripts.glove2word2vec import glove2word2vec

# The text file you just unzipped
glove_input_file = 'dolma_300_2024_1.2M.100_combined.txt' 

# The new optimized file you are creating
word2vec_output_file = 'glove.840B.300d.word2vec' 

print("Converting... this will take a few minutes and high RAM.")
glove2word2vec(glove_input_file, word2vec_output_file)
print("Done! You can now use the output file in the AffinityEngine.")

Converting... this will take a few minutes and high RAM.


C:\Users\dzhan\AppData\Local\Temp\ipykernel_8132\3440702572.py:10: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)


Done! You can now use the output file in the AffinityEngine.


In [ ]:
# --- Configuration ---
CSV_FILEPATH = 'output.csv' # Replace with your actual file name
GLOVE_WORD2VEC_PATH = 'glove.840B.300d.word2vec' # Your converted gensim model
ALPHA_PENALTY = 0.1 # Try toggling this to see how it affects the matrix

# --- 1. Load Data ---
print("Parsing historical games...")
parsed_games = load_puzzles_from_csv(CSV_FILEPATH)
print(f"Successfully loaded {len(parsed_games)} puzzles.\n")

# engine = AffinityEngine(model_path=GLOVE_WORD2VEC_PATH, alpha=ALPHA_PENALTY)
# Grab the first puzzle to test
# test_puzzle = parsed_games[0]
# board_words = test_puzzle.raw_board
# print(f"\nEvaluating Puzzle #{test_puzzle.puzzle_id}")
# print(f"Board State: {board_words}\n")
# # Generate the affinity matrix
# affinity_matrix = engine.build_base_matrix(board_words)
# ground_truth = test_puzzle.get_ground_truth_matrix()
# # Print a small diagnostic sample (e.g., how Word 0 relates to Word 1 vs Word 5)
# print(f"Diagnostic Sample for '{board_words[0]}':")
# print(f"  Affinity w/ '{board_words[1]}' (Same Group): {affinity_matrix[0, 1]:.3f} (True: {ground_truth[0, 1]})")
# print(f"  Affinity w/ '{board_words[5]}' (Diff Group): {affinity_matrix[0, 5]:.3f} (True: {ground_truth[0, 5]})")
# print(f"  Affinity w/ '{board_words[15]}' (Diff Group): {affinity_matrix[0, 15]:.3f} (True: {ground_truth[0, 15]})")
# print(f"\nGenerated Matrix Shape: {affinity_matrix.shape}")

Parsing historical games...
Successfully loaded 915 puzzles.

Loading embeddings from glove.840B.300d.word2vec into memory...
Embeddings successfully loaded.

Evaluating Puzzle #1
Board State: ['curses', 'fudge', 'blast', 'crud', 'choral', 'jazz', 'rap', 'americana', 'lord', 'please', 'sheesh', 'brother', 'heavens', 'gracious', 'mercy', 'dear']

Diagnostic Sample for 'curses':
  Affinity w/ 'fudge' (Same Group): 0.000 (True: 1)
  Affinity w/ 'jazz' (Diff Group): 0.000 (True: 0)
  Affinity w/ 'dear' (Diff Group): 0.000 (True: 0)

Generated Matrix Shape: (16, 16)


In [10]:
import nltk
import jellyfish
import numpy as np
from typing import List, Dict

# Download WordNet data (only runs if not already downloaded)
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Downloading WordNet corpora...")
    nltk.download('wordnet')
    
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dzhan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
from gensim.models import KeyedVectors
import numpy as np
import jellyfish
from nltk.corpus import wordnet as wn
from typing import List

class AffinityEngine:
    def __init__(self, 
                 model_path: str, 
                 weights: tuple = (0.6, 0.3, 0.05), 
                 alpha: float = 0.1):
        
        print(f"Loading embeddings from {model_path} into memory...")
        self.model = KeyedVectors.load_word2vec_format(model_path, binary=False)
        self.vector_size = self.model.vector_size
        self.weights = np.array(weights) / sum(weights)
        self.alpha = alpha
        print("Affinity Engine fully initialized.")

    def extract_semantic_matrix(self, words: List[str]) -> np.ndarray:
        vectors = np.array([self.model[w] if w in self.model else np.zeros(self.vector_size) for w in words])
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1 
        normalized = vectors / norms
        sim_matrix = np.dot(normalized, normalized.T)
        return np.clip((sim_matrix + 1) / 2, 0.0, 1.0)

    def extract_structural_matrix(self, words: List[str]) -> np.ndarray:
        n = len(words)
        matrix = np.zeros((n, n))
        synsets = {w: wn.synsets(w) for w in words}
        
        for i in range(n):
            for j in range(n):
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                
                syns1 = synsets[words[i]]
                syns2 = synsets[words[j]]
                
                if not syns1 or not syns2:
                    continue
                
                max_sim = 0.0
                for s1 in syns1:
                    for s2 in syns2:
                        sim = s1.wup_similarity(s2)
                        if sim and sim > max_sim:
                            max_sim = sim
                            
                matrix[i, j] = max_sim
        return matrix

    def extract_orthographic_matrix(self, words: List[str]) -> np.ndarray:
        n = len(words)
        matrix = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                    
                w1, w2 = words[i], words[j]
                score = 0.0
                
                if jellyfish.soundex(w1) == jellyfish.soundex(w2): score += 0.8 
                if len(w1) > 3 and len(w2) > 3 and w1[-3:] == w2[-3:]: score += 0.5
                if w1[:3] == w2[:3]: score += 0.5
                if sorted(w1) == sorted(w2): score += 0.9

                matrix[i, j] = min(score, 1.0)
        return matrix

    def build_base_matrix(self, words: List[str]) -> np.ndarray:
        n = len(words)
        m_sem = self.extract_semantic_matrix(words)
        m_str = self.extract_structural_matrix(words)
        m_ort = self.extract_orthographic_matrix(words)
        
        fused_matrix = (
            (self.weights[0] * m_sem) + 
            (self.weights[1] * m_str) + 
            (self.weights[2] * m_ort)
        )
        
        if self.alpha > 0:
            row_sums = fused_matrix.sum(axis=1)
            for i in range(n):
                for j in range(n):
                    penalty = self.alpha * (row_sums[i] + row_sums[j] - 2 * fused_matrix[i, j])
                    fused_matrix[i, j] -= penalty
                    
        return np.clip(fused_matrix, 0.0, 1.0)

In [ ]:
test_puzzle = parsed_games[0]
board_words = test_puzzle.raw_board

# Initialize the completed engine (You can tweak these weights later!)
# engine = AffinityEngine(
#     model_path=GLOVE_WORD2VEC_PATH, 
#     weights=(0.60, 0.25, 0.15), 
#     alpha=0.1
# )

# Generate the master affinity matrix
# master_matrix = engine.build_base_matrix(board_words)
# ground_truth = test_puzzle.get_ground_truth_matrix()

# print(f"\nSuccessfully generated {master_matrix.shape} master matrix for Puzzle #{test_puzzle.puzzle_id}")
# print("\nSample Output (Word 0 vs Word 1 - Should be same group):")
# print(f"Fused Affinity: {master_matrix[0, 1]:.3f} (True: {ground_truth[0, 1]})")

Loading embeddings from glove.840B.300d.word2vec into memory...
Affinity Engine fully initialized.


In [37]:
import itertools
from typing import List, Tuple
import numpy as np

class PartitionSolver:
    def __init__(self, affinity_matrix: np.ndarray, words: List[str], previous_guesses: List[Tuple[List[str], str]] = None, max_iters: int = 5000):
        self.matrix = affinity_matrix
        self.words = [w.lower() for w in words]
        self.n_words = len(self.words)
        self.target_groups = self.n_words // 4
        self.max_iters = max_iters  # NEW: The impatience limit
        self.current_iters = 0      # NEW: The iteration counter
        
        self.previous_guesses = previous_guesses or []
        self.guess_constraints = []
        for guess_words, status in self.previous_guesses:
            guess_words_lower = [w.lower() for w in guess_words]
            g_mask = sum(1 << self.words.index(w) for w in guess_words_lower if w in self.words)
            self.guess_constraints.append((g_mask, status))
            
    def score_group(self, indices: Tuple[int, ...]) -> float:
        score = 0.0
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                score += self.matrix[indices[i], indices[j]]
        return score

    def solve(self) -> List[Tuple[List[str], float]]:
        groups = []
        
        for combo in itertools.combinations(range(self.n_words), 4):
            mask = sum(1 << i for i in combo)
            if any(mask == g_mask for g_mask, _ in self.guess_constraints):
                continue 
            score = self.score_group(combo)
            groups.append((score, mask, combo))

        groups.sort(key=lambda x: x[0], reverse=True)

        best_partition = None
        best_score = -float('inf')

        def dfs(group_idx: int, current_partition: list, current_score: float, used_mask: int):
            nonlocal best_score, best_partition
            
            # NEW: The Kill Switch
            self.current_iters += 1
            if self.current_iters > self.max_iters:
                return 

            if len(current_partition) == self.target_groups:
                valid_board = True
                for g_mask, status in self.guess_constraints:
                    if status == "ONE_AWAY":
                        satisfied = False
                        for _, p_mask, _ in current_partition:
                            if bin(p_mask & g_mask).count('1') == 3:
                                satisfied = True
                                break
                        if not satisfied:
                            valid_board = False
                            break
                
                if valid_board and current_score > best_score:
                    best_score = current_score
                    best_partition = current_partition[:]
                return

            if group_idx < len(groups):
                max_possible = current_score + (self.target_groups - len(current_partition)) * groups[group_idx][0]
                if max_possible <= best_score:
                    return

            for i in range(group_idx, len(groups)):
                score, mask, combo = groups[i]
                if (used_mask & mask) == 0:
                    current_partition.append(groups[i])
                    dfs(i + 1, current_partition, current_score + score, used_mask | mask)
                    current_partition.pop()

        dfs(0, [], 0.0, 0)

        result = []
        if best_partition:
            for score, mask, combo in best_partition:
                group_words = [self.words[idx] for idx in combo]
                result.append((group_words, score))
            
        return result

In [47]:
class GameHarness:
    def __init__(self, engine, words: list, answer_key: list = None):
        self.engine = engine
        
        # Save the original 16 words to act as the index map
        self.original_words = [w.lower() for w in words]
        self.active_words = self.original_words[:]
        
        # BUILD THE MATRIX ONCE
        self.master_matrix = self.engine.build_base_matrix(self.original_words)
        
        self.answer_key = [[w.lower() for w in cat] for cat in answer_key] if answer_key else None
        self.history = []        
        self.solved_groups = []  
        self.attempts = 0        
        self.mistakes = 0

    def get_feedback(self, guess: list) -> str:
        if self.answer_key:
            guess_set = set(guess)
            for cat in self.answer_key:
                overlap = len(guess_set.intersection(set(cat)))
                if overlap == 4: return "CORRECT"
                if overlap == 3: return "ONE_AWAY"
            return "WRONG"
        else:
            print(f"\n>>> SOLVER GUESSES: {guess}")
            while True:
                resp = input("Feedback? (1: Correct, 2: One Away, 3: Wrong): ").strip()
                if resp == '1': return "CORRECT"
                if resp == '2': return "ONE_AWAY"
                if resp == '3': return "WRONG"
                print("Invalid input. Please enter 1, 2, or 3.")

    def play(self):
        print(f"--- STARTING GAME WITH {len(self.active_words)} WORDS ---")
        
        while len(self.active_words) > 0:
            self.attempts += 1
            
            # THE FIX: Slice the pre-calculated master matrix instead of rebuilding it
            idx_map = [self.original_words.index(w) for w in self.active_words]
            current_matrix = self.master_matrix[np.ix_(idx_map, idx_map)]
            
            solver = PartitionSolver(current_matrix, self.active_words, self.history)
            best_partitions = solver.solve()
            
            if not best_partitions:
                print("CRITICAL ERROR: No mathematically valid partitions found.")
                break
                
            top_guess = best_partitions[0][0] 
            status = self.get_feedback(top_guess)
            
            if self.answer_key:
                 print(f"Attempt {self.attempts} | Guess: {top_guess} -> {status}")
            
            if status == "CORRECT":
                self.solved_groups.append(top_guess)
                self.active_words = [w for w in self.active_words if w not in top_guess]
                self.history = [] 
                print(f"✅ GROUP SECURED! Remaining words: {len(self.active_words)}")
                
            else:
                self.mistakes += 1
                self.history.append((top_guess, status))
                if status == "ONE_AWAY":
                    print("⚠️ ONE AWAY! Solver is updating its global constraints...")
                else:
                    print("❌ WRONG! Solver is pruning this branch.")
                    
            if self.mistakes >= 4 and not self.answer_key:
                print("\n💀 GAME OVER: 4 mistakes reached. (Testing continues).")
                
        print("\n--- GAME COMPLETE ---")
        print(f"Total Attempts: {self.attempts}")
        for i, group in enumerate(self.solved_groups):
            print(f"Final Group {i+1}: {group}")

In [34]:
import numpy as np
from typing import List, Dict

# Assuming 'parsed_games' and 'engine' are already loaded from previous cells

print(f"Pre-calculating raw matrices for {len(parsed_games)} games...")
print("This will take a few minutes, but only needs to run ONCE.")

cached_dataset = []

for puzzle in parsed_games:
    words = puzzle.raw_board
    
    # Extract the raw tracks independently
    m_sem = engine.extract_semantic_matrix(words)
    m_str = engine.extract_structural_matrix(words)
    m_ort = engine.extract_orthographic_matrix(words)
    
    # Save to memory
    cached_dataset.append({
        'puzzle_id': puzzle.puzzle_id,
        'words': words,
        'true_categories': [cat.words for cat in puzzle.categories],
        'm_sem': m_sem,
        'm_str': m_str,
        'm_ort': m_ort
    })

print("✅ All matrices cached! Ready for instant Grid Search.")

Pre-calculating raw matrices for 915 games...
This will take a few minutes, but only needs to run ONCE.
✅ All matrices cached! Ready for instant Grid Search.


In [35]:
def evaluate_weights(cached_game: dict, weights: tuple, alpha: float) -> int:
    """Fuses the cached matrices with test weights and scores the solver."""
    w_sem, w_str, w_ort = weights
    n = len(cached_game['words'])
    
    # 1. Fast Matrix Fusion
    fused_matrix = (w_sem * cached_game['m_sem']) + \
                   (w_str * cached_game['m_str']) + \
                   (w_ort * cached_game['m_ort'])
    
    # 2. Apply Alpha Penalty
    if alpha > 0:
        row_sums = fused_matrix.sum(axis=1)
        for i in range(n):
            for j in range(n):
                penalty = alpha * (row_sums[i] + row_sums[j] - 2 * fused_matrix[i, j])
                fused_matrix[i, j] -= penalty
                
    fused_matrix = np.clip(fused_matrix, 0.0, 1.0)
    
    # 3. Simulate the Game (Stripped down GameHarness logic for speed)
    active_words = cached_game['words'][:]
    history = []
    mistakes = 0
    groups_solved = 0
    
    while len(active_words) > 0 and mistakes < 4:
        # Subset the matrix for remaining words
        idx_map = [cached_game['words'].index(w) for w in active_words]
        current_matrix = fused_matrix[np.ix_(idx_map, idx_map)]
        
        solver = PartitionSolver(current_matrix, active_words, history)
        best_partitions = solver.solve()
        
        if not best_partitions:
            break
            
        top_guess = best_partitions[0][0]
        guess_set = set(top_guess)
        
        # Check against answer key
        status = "WRONG"
        for cat in cached_game['true_categories']:
            overlap = len(guess_set.intersection(set(cat)))
            if overlap == 4:
                status = "CORRECT"
                break
            elif overlap == 3:
                status = "ONE_AWAY"
                
        if status == "CORRECT":
            groups_solved += 1
            active_words = [w for w in active_words if w not in top_guess]
            history = [] # Reset history on success
        else:
            mistakes += 1
            history.append((top_guess, status))
            
    return groups_solved

In [ ]:
# --- DEFINE THE SEARCH SPACE ---

valid_weights = []

# Generate fine-tuned percentages (using integers to avoid float math bugs)
# We test Semantic from 60% to 90%, and the others from 0% to 30% in 5% increments.
for sem in range(60, 95, 5):      
    for struct in range(0, 35, 5):  
        for ortho in range(0, 35, 5): 
            if sem + struct + ortho == 100:
                # Convert back to standard decimal weights
                valid_weights.append((sem/100, struct/100, ortho/100))

alpha_options = [0.05, 0.08, 0.10, 0.12]

print(f"Testing {len(valid_weights)} weight profiles across {len(alpha_options)} alpha values.")
print(f"Total Configurations: {len(valid_weights) * len(alpha_options)}\n")

# --- RUN THE GRID SEARCH ---
best_config = None
best_avg_score = -1

for alpha in alpha_options:
    for weights in valid_weights:
        total_score = 0
        
        # Test against the full dataset
        test_subset = cached_dataset[:] 
        
        for game in test_subset:
            score = evaluate_weights(game, weights, alpha)
            total_score += score
            
        avg_score = total_score / len(test_subset)
        
        print(f"Weights (Sem:{weights[0]:.2f}, Str:{weights[1]:.2f}, Ort:{weights[2]:.2f}) | Alpha: {alpha:.2f} -> Avg Score: {avg_score:.2f} / 4.0")
        
        if avg_score > best_avg_score:
            best_avg_score = avg_score
            best_config = (weights, alpha)

print("\nOPTIMIZATION COMPLETE")
print(f"Best Average Score: {best_avg_score:.2f} / 4.0")
print(f"Optimal Weights (Semantic, Structural, Orthographic): {best_config[0]}")
print(f"Optimal Alpha Penalty: {best_config[1]}")

Testing 36 weight profiles across 4 alpha values.
Total Configurations: 144

Weights (Sem:0.60, Str:0.10, Ort:0.30) | Alpha: 0.05 -> Avg Score: 2.57 / 4.0
Weights (Sem:0.60, Str:0.15, Ort:0.25) | Alpha: 0.05 -> Avg Score: 2.71 / 4.0
Weights (Sem:0.60, Str:0.20, Ort:0.20) | Alpha: 0.05 -> Avg Score: 2.73 / 4.0
Weights (Sem:0.60, Str:0.25, Ort:0.15) | Alpha: 0.05 -> Avg Score: 2.73 / 4.0
Weights (Sem:0.60, Str:0.30, Ort:0.10) | Alpha: 0.05 -> Avg Score: 2.72 / 4.0
Weights (Sem:0.65, Str:0.05, Ort:0.30) | Alpha: 0.05 -> Avg Score: 2.64 / 4.0
Weights (Sem:0.65, Str:0.10, Ort:0.25) | Alpha: 0.05 -> Avg Score: 2.75 / 4.0
Weights (Sem:0.65, Str:0.15, Ort:0.20) | Alpha: 0.05 -> Avg Score: 2.80 / 4.0
Weights (Sem:0.65, Str:0.20, Ort:0.15) | Alpha: 0.05 -> Avg Score: 2.80 / 4.0
Weights (Sem:0.65, Str:0.25, Ort:0.10) | Alpha: 0.05 -> Avg Score: 2.85 / 4.0
Weights (Sem:0.65, Str:0.30, Ort:0.05) | Alpha: 0.05 -> Avg Score: 2.88 / 4.0
Weights (Sem:0.70, Str:0.00, Ort:0.30) | Alpha: 0.05 -> Avg Score

In [44]:
# with optimal weights
engine = AffinityEngine(
    model_path=GLOVE_WORD2VEC_PATH, 
    weights=(0.65, 0.30, 0.05), 
    alpha=0.08
)

Loading embeddings from glove.840B.300d.word2vec into memory...
Affinity Engine fully initialized.


In [48]:
custom_categories = [
    # Difficulty 1 (Yellow) - usually straightforward semantic links
    Category(label="FISH", words=["bass", "trout", "salmon", "flounder"], difficulty=1),
    
    # Difficulty 2 (Green) - semantics with some slight ambiguity
    Category(label="INSTRUMENTS", words=["guitar", "drum", "piano", "flute"], difficulty=2),
    
    # Difficulty 3 (Blue) - often entity/pop-culture based
    Category(label="FRUITS", words=["apple", "banana", "orange", "pear"], difficulty=3),
    
    # Difficulty 4 (Purple) - usually wordplay, fill-in-the-blank, or spelling tricks
    Category(label="PETS", words=["cat", "dog", "mouse", "bird"], difficulty=4)
]

# --- ASSEMBLE THE GAME STATE ---
custom_puzzle = ConnectionsPuzzle(puzzle_id=999, categories=custom_categories)

# Extract the flat 16-word list for the engine
custom_board = custom_puzzle.raw_board

# Extract the nested lists for the auto-evaluator
custom_answer_key = [cat.words for cat in custom_puzzle.categories]

print(f"Board Words: {custom_board}")

custom_harness = GameHarness(engine, custom_board, answer_key=None)

# Let it play!
custom_harness.play()

Board Words: ['bass', 'trout', 'salmon', 'flounder', 'guitar', 'drum', 'piano', 'flute', 'apple', 'banana', 'orange', 'pear', 'cat', 'dog', 'mouse', 'bird']
--- STARTING GAME WITH 16 WORDS ---

>>> SOLVER GUESSES: ['bass', 'trout', 'salmon', 'flounder']
✅ GROUP SECURED! Remaining words: 12

>>> SOLVER GUESSES: ['guitar', 'drum', 'piano', 'flute']
✅ GROUP SECURED! Remaining words: 8

>>> SOLVER GUESSES: ['apple', 'banana', 'orange', 'pear']
✅ GROUP SECURED! Remaining words: 4

>>> SOLVER GUESSES: ['cat', 'dog', 'mouse', 'bird']
✅ GROUP SECURED! Remaining words: 0

--- GAME COMPLETE ---
Total Attempts: 4
Final Group 1: ['bass', 'trout', 'salmon', 'flounder']
Final Group 2: ['guitar', 'drum', 'piano', 'flute']
Final Group 3: ['apple', 'banana', 'orange', 'pear']
Final Group 4: ['cat', 'dog', 'mouse', 'bird']


In [49]:
import time
import numpy as np

print(f"Starting Full Archive Evaluation on {len(parsed_games)} games...")
start_time = time.time()

total_games = len(parsed_games)
total_score = 0
perfect_games = 0

for idx, puzzle in enumerate(parsed_games):
    # 1. Build the static master matrix for this specific game
    master_matrix = engine.build_base_matrix(puzzle.raw_board)
    
    # 2. Setup the game state
    active_words = puzzle.raw_board[:]
    true_categories = [cat.words for cat in puzzle.categories]
    history = []
    mistakes = 0
    groups_solved = 0
    
    # 3. The Game Loop (Silent Mode)
    while len(active_words) > 0 and mistakes < 4:
        # Slice the pre-calculated matrix for the remaining words
        idx_map = [puzzle.raw_board.index(w) for w in active_words]
        current_matrix = master_matrix[np.ix_(idx_map, idx_map)]
        
        solver = PartitionSolver(current_matrix, active_words, history)
        best_partitions = solver.solve()
        
        if not best_partitions:
            break  # Math error, skip to next game
            
        top_guess = best_partitions[0][0]
        guess_set = set(top_guess)
        
        # Check against answer key
        status = "WRONG"
        for cat in true_categories:
            overlap = len(guess_set.intersection(set(cat)))
            if overlap == 4:
                status = "CORRECT"
                break
            elif overlap == 3:
                status = "ONE_AWAY"
                
        # Process Feedback
        if status == "CORRECT":
            groups_solved += 1
            active_words = [w for w in active_words if w not in top_guess]
            history = [] # Reset constraints on success
        else:
            mistakes += 1
            history.append((top_guess, status))
            
    # 4. Tally Final Score for this game
    total_score += groups_solved
    
    # In Connections, solving 3 means you automatically get the 4th. 
    # We will count a 4/4 as a "Perfect Win" 
    if groups_solved == 4:
        perfect_games += 1
        
    # Print a progress update so you know it hasn't frozen
    if (idx + 1) % 50 == 0 or (idx + 1) == total_games:
        print(f"Processed {idx + 1}/{total_games} games...")

end_time = time.time()

# --- FINAL REPORT ---
print("\nFULL ARCHIVE EVALUATION COMPLETE")
print(f"Total Games Played: {total_games}")
print(f"Average Groups Found: {total_score / total_games:.2f} / 4.0")
print(f"Perfect Wins (Cleared the Board): {perfect_games} ({(perfect_games/total_games)*100:.1f}%)")
print(f"Time Taken: {end_time - start_time:.2f} seconds")

Starting Full Archive Evaluation on 915 games...
Processed 50/915 games...
Processed 100/915 games...
Processed 150/915 games...
Processed 200/915 games...
Processed 250/915 games...
Processed 300/915 games...
Processed 350/915 games...
Processed 400/915 games...
Processed 450/915 games...
Processed 500/915 games...
Processed 550/915 games...
Processed 600/915 games...
Processed 650/915 games...
Processed 700/915 games...
Processed 750/915 games...
Processed 800/915 games...
Processed 850/915 games...
Processed 900/915 games...
Processed 915/915 games...

FULL ARCHIVE EVALUATION COMPLETE
Total Games Played: 915
Average Groups Found: 3.39 / 4.0
Perfect Wins (Cleared the Board): 654 (71.5%)
Time Taken: 624.64 seconds
